
# LSTM - AESO Price Spike Forecasting (Local Notebook)

This notebook is a local version of the Colab notebook you shared.

Target:
- `spike_lead_2` = 1 when the actual pool price at `t+2` is above `$200/MWh`

Why F2 here:
- the spike class is rare
- false negatives matter more than false positives
- F2 puts more weight on recall than F1, so it is a better primary tuning metric for this task

Notebook design:
- local file paths, no `google.colab`
- Windows-safe dataloaders (`num_workers=0`)
- strict train / validation / test workflow
- simple structure so it is easy to present in class


In [ ]:

# 1. Imports and configuration
import json
import random
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import fbeta_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

optuna.logging.set_verbosity(optuna.logging.WARNING)


def find_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "Data" / "CSVs" / "aeso_merged_2020_2025.csv").exists() and (candidate / "JorgeFolder").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
SOURCE_DATA = PROJECT_ROOT / "Data" / "CSVs" / "aeso_merged_2020_2025.csv"
NOTEBOOK_DIR = PROJECT_ROOT / "JorgeFolder" / "models" / "lstm"
OUTPUT_DIR = NOTEBOOK_DIR / "notebook_run"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SPIKE_THRESHOLD = 200.0
TARGET = "spike_lead_2"
TRAIN_END_EXCLUSIVE = pd.Timestamp("2023-11-06 00:00:00")
VAL_END_EXCLUSIVE = pd.Timestamp("2024-12-12 00:00:00")
TEST_START = VAL_END_EXCLUSIVE
TS_CV_SPLITS = 5
N_OPTUNA_TRIALS = 5
MAX_EPOCHS = 20
PATIENCE = 5
THRESHOLD = 0.5
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()
RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Project root : {PROJECT_ROOT}")
print(f"Source data  : {SOURCE_DATA}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Device       : {DEVICE}")


In [ ]:

# 2. Feature list
FEATURES = [
    "ACTUAL_POOL_PRICE",
    "ACTUAL_AIL",
    "gas_total",
    "wind_total",
    "solar_total",
    "coal_total",
    "hydro_total",
    "net_export",
    "IMPORT_BC",
    "IMPORT_MT",
    "IMPORT_SK",
    "net_load",
    "net_load_3h_change",
    "reserve_margin",
    "resilience_buffer",
    "renewables_share",
    "dispatchable_ratio",
    "gas_ratio",
    "renewable_generation",
    "dispatchable_generation",
    "sin_hour",
    "cos_hour",
    "sin_dow",
    "cos_dow",
    "sin_year",
    "cos_year",
    "is_weekend",
    "is_stampede",
]

NO_SCALE = [
    "sin_hour", "cos_hour", "sin_dow", "cos_dow", "sin_year", "cos_year",
    "is_weekend", "is_stampede",
]
CONTINUOUS = [column for column in FEATURES if column not in NO_SCALE]

BASE_REQUIRED_COLUMNS = [
    "datetime", "ACTUAL_POOL_PRICE", "ACTUAL_AIL", "gas_total", "wind_total", "solar_total",
    "coal_total", "hydro_total", "net_export", "IMPORT_BC", "IMPORT_MT", "IMPORT_SK",
    "net_load", "net_load_3h_change", "reserve_margin", "resilience_buffer",
    "renewables_share", "dispatchable_ratio", "gas_ratio",
    "renewable_generation", "dispatchable_generation",
]

print(f"Feature count: {len(FEATURES)}")


In [ ]:

# 3. Load data and build the modeling target locally
source_df = pd.read_csv(SOURCE_DATA, parse_dates=["datetime"])
source_df = source_df.sort_values("datetime").reset_index(drop=True)

missing_columns = sorted(set(BASE_REQUIRED_COLUMNS) - set(source_df.columns))
if missing_columns:
    raise ValueError(f"Source data is missing required columns: {missing_columns}")

df = source_df.copy()

# Remove leap day so yearly cyclical encoding stays on a 365-day cycle.
df = df[~((df["datetime"].dt.month == 2) & (df["datetime"].dt.day == 29))].reset_index(drop=True)

if "is_weekend" not in df.columns:
    df["is_weekend"] = (df["datetime"].dt.dayofweek >= 5).astype(int)

if "is_stampede" not in df.columns:
    df["is_stampede"] = 0
    for start, end in [
        ("2021-07-09", "2021-07-18"),
        ("2022-07-08", "2022-07-17"),
        ("2023-07-07", "2023-07-16"),
        ("2024-07-05", "2024-07-14"),
        ("2025-07-04", "2025-07-13"),
    ]:
        df.loc[df["datetime"].between(start, end), "is_stampede"] = 1

doy = df["datetime"].dt.day_of_year.astype(float)
df["sin_hour"] = np.sin(2 * np.pi * df["datetime"].dt.hour / 24)
df["cos_hour"] = np.cos(2 * np.pi * df["datetime"].dt.hour / 24)
df["sin_dow"] = np.sin(2 * np.pi * df["datetime"].dt.dayofweek / 7)
df["cos_dow"] = np.cos(2 * np.pi * df["datetime"].dt.dayofweek / 7)
df["sin_year"] = np.sin(2 * np.pi * doy / 365)
df["cos_year"] = np.cos(2 * np.pi * doy / 365)

df["pool_price_lead_2"] = df["ACTUAL_POOL_PRICE"].shift(-2)
df[TARGET] = (df["pool_price_lead_2"] > SPIKE_THRESHOLD).astype("Int64")

df = df.dropna(subset=["datetime", *FEATURES, "pool_price_lead_2", TARGET]).reset_index(drop=True)
df[TARGET] = df[TARGET].astype(int)

print(f"Rows after cleaning : {len(df):,}")
print(f"Overall spike rate  : {df[TARGET].mean():.4f}")
df[["datetime", "ACTUAL_POOL_PRICE", "pool_price_lead_2", TARGET]].head()


In [ ]:

# 4. Strict train / validation / test split
train = df[df["datetime"] < TRAIN_END_EXCLUSIVE].reset_index(drop=True)
validation = df[(df["datetime"] >= TRAIN_END_EXCLUSIVE) & (df["datetime"] < VAL_END_EXCLUSIVE)].reset_index(drop=True)
test = df[df["datetime"] >= TEST_START].reset_index(drop=True)
trainval = df[df["datetime"] < TEST_START].reset_index(drop=True)

split_summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train),
        "start": train["datetime"].min(),
        "end": train["datetime"].max(),
        "spike_rate": round(float(train[TARGET].mean()), 4),
    },
    {
        "split": "validation",
        "rows": len(validation),
        "start": validation["datetime"].min(),
        "end": validation["datetime"].max(),
        "spike_rate": round(float(validation[TARGET].mean()), 4),
    },
    {
        "split": "test",
        "rows": len(test),
        "start": test["datetime"].min(),
        "end": test["datetime"].max(),
        "spike_rate": round(float(test[TARGET].mean()), 4),
    },
])

print("Approximate class imbalance by split:")
for row in split_summary.itertuples(index=False):
    print(f"  {row.split:10s} rows={row.rows:6d}  spike_rate={row.spike_rate:.4f}")

split_summary


In [ ]:

# 5. Sequence builder and LSTM model

def make_sequences(x_matrix, y_array, indices, seq_len):
    sequences, labels, used_indices = [], [], []
    for idx in indices:
        start_idx = idx - seq_len + 1
        if start_idx < 0:
            continue
        sequences.append(x_matrix[start_idx : idx + 1])
        labels.append(y_array[idx])
        used_indices.append(idx)
    return (
        np.asarray(sequences, dtype=np.float32),
        np.asarray(labels, dtype=np.float32),
        np.asarray(used_indices, dtype=np.int64),
    )


class SeqDataset(Dataset):
    def __init__(self, x_values, y_values):
        self.x_values = torch.tensor(x_values, dtype=torch.float32)
        self.y_values = torch.tensor(y_values, dtype=torch.float32)

    def __len__(self):
        return len(self.y_values)

    def __getitem__(self, index):
        return self.x_values[index], self.y_values[index]


class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, inputs):
        output, _ = self.lstm(inputs)
        return self.fc(self.dropout(output[:, -1, :])).squeeze(-1)


In [ ]:

# 6. Optuna tuning with F2 as the direct objective
trainval_targets = trainval[TARGET].to_numpy(dtype=np.float32)
folds = list(TimeSeriesSplit(n_splits=TS_CV_SPLITS).split(trainval))
tune_train_idx, tune_val_idx = folds[-1]


def optuna_objective(trial):
    seq_len = trial.suggest_categorical("seq_len", [12, 24, 48])
    hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 96, 128])
    num_layers = trial.suggest_categorical("num_layers", [1, 2])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    scaler = StandardScaler()
    scaler.fit(trainval.iloc[tune_train_idx][CONTINUOUS])
    x_trainval = trainval[FEATURES].copy().astype({column: float for column in CONTINUOUS}, copy=False)
    x_trainval.loc[:, CONTINUOUS] = scaler.transform(x_trainval[CONTINUOUS])
    x_trainval = x_trainval.to_numpy(dtype=np.float32)

    x_train, y_train, _ = make_sequences(x_trainval, trainval_targets, tune_train_idx, seq_len)
    x_val, y_val, _ = make_sequences(x_trainval, trainval_targets, tune_val_idx, seq_len)

    train_loader = DataLoader(
        SeqDataset(x_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        SeqDataset(x_val, y_val),
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    positives = float(y_train.sum())
    negatives = float(len(y_train) - positives)
    pos_weight = torch.tensor([negatives / max(positives, 1.0)], dtype=torch.float32, device=DEVICE)

    model = LSTMClassifier(len(FEATURES), hidden_size, num_layers, dropout).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_f2 = -1.0
    patience_counter = 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(x_batch.to(DEVICE))
            loss = criterion(logits, y_batch.to(DEVICE))
            loss.backward()
            optimizer.step()

        model.eval()
        probabilities = []
        with torch.no_grad():
            for x_batch, _ in val_loader:
                probabilities.append(torch.sigmoid(model(x_batch.to(DEVICE))).cpu().numpy())
        probabilities = np.concatenate(probabilities)
        f2 = fbeta_score(y_val, (probabilities >= THRESHOLD).astype(int), beta=2, zero_division=0)

        if f2 > best_f2:
            best_f2 = float(f2)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    return best_f2


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(optuna_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)

BEST = study.best_params
print("Best params:")
print(BEST)
print(f"Best tuning F2: {study.best_value:.4f}")


In [ ]:

# 7. Full TimeSeriesSplit cross-validation using F2
cv_rows = []

for fold_number, (cv_train_idx, cv_val_idx) in enumerate(folds, start=1):
    scaler = StandardScaler()
    scaler.fit(trainval.iloc[cv_train_idx][CONTINUOUS])
    x_trainval = trainval[FEATURES].copy().astype({column: float for column in CONTINUOUS}, copy=False)
    x_trainval.loc[:, CONTINUOUS] = scaler.transform(x_trainval[CONTINUOUS])
    x_trainval = x_trainval.to_numpy(dtype=np.float32)

    x_train, y_train, _ = make_sequences(x_trainval, trainval_targets, cv_train_idx, BEST["seq_len"])
    x_val, y_val, _ = make_sequences(x_trainval, trainval_targets, cv_val_idx, BEST["seq_len"])

    train_loader = DataLoader(
        SeqDataset(x_train, y_train),
        batch_size=BEST["batch_size"],
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        SeqDataset(x_val, y_val),
        batch_size=BEST["batch_size"] * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    positives = float(y_train.sum())
    negatives = float(len(y_train) - positives)
    pos_weight = torch.tensor([negatives / max(positives, 1.0)], dtype=torch.float32, device=DEVICE)

    model = LSTMClassifier(len(FEATURES), BEST["hidden_size"], BEST["num_layers"], BEST["dropout"]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=BEST["lr"], weight_decay=BEST["weight_decay"])

    best_f2 = -1.0
    best_epoch = None
    patience_counter = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(x_batch.to(DEVICE))
            loss = criterion(logits, y_batch.to(DEVICE))
            loss.backward()
            optimizer.step()

        model.eval()
        probabilities = []
        with torch.no_grad():
            for x_batch, _ in val_loader:
                probabilities.append(torch.sigmoid(model(x_batch.to(DEVICE))).cpu().numpy())
        probabilities = np.concatenate(probabilities)
        f2 = fbeta_score(y_val, (probabilities >= THRESHOLD).astype(int), beta=2, zero_division=0)

        if f2 > best_f2:
            best_f2 = float(f2)
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    cv_rows.append(
        {
            "fold": fold_number,
            "best_f2": round(float(best_f2), 4),
            "best_epoch": int(best_epoch),
            "train_rows": int(len(y_train)),
            "validation_rows": int(len(y_val)),
            "train_spike_rate": round(float(y_train.mean()), 4),
            "validation_spike_rate": round(float(y_val.mean()), 4),
        }
    )
    print(f"Fold {fold_number}/{TS_CV_SPLITS}: best F2 = {best_f2:.4f} at epoch {best_epoch}")

cv_results = pd.DataFrame(cv_rows)
print(f"Mean CV F2: {cv_results['best_f2'].mean():.4f} +/- {cv_results['best_f2'].std(ddof=0):.4f}")
cv_results


In [ ]:

# 8. Final model: early stopping on validation, final evaluation on untouched test
full = pd.concat([train, validation, test], ignore_index=True)
full_targets = full[TARGET].to_numpy(dtype=np.float32)

train_idx = np.flatnonzero(full["datetime"] < TRAIN_END_EXCLUSIVE)
val_idx = np.flatnonzero((full["datetime"] >= TRAIN_END_EXCLUSIVE) & (full["datetime"] < VAL_END_EXCLUSIVE))
test_idx = np.flatnonzero(full["datetime"] >= TEST_START)

scaler_final = StandardScaler()
scaler_final.fit(train[CONTINUOUS])
x_full = full[FEATURES].copy().astype({column: float for column in CONTINUOUS}, copy=False)
x_full.loc[:, CONTINUOUS] = scaler_final.transform(x_full[CONTINUOUS])
x_full = x_full.to_numpy(dtype=np.float32)

x_train, y_train, train_used_idx = make_sequences(x_full, full_targets, train_idx, BEST["seq_len"])
x_val, y_val, val_used_idx = make_sequences(x_full, full_targets, val_idx, BEST["seq_len"])
x_test, y_test, test_used_idx = make_sequences(x_full, full_targets, test_idx, BEST["seq_len"])

train_loader = DataLoader(
    SeqDataset(x_train, y_train),
    batch_size=BEST["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    SeqDataset(x_val, y_val),
    batch_size=BEST["batch_size"] * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    SeqDataset(x_test, y_test),
    batch_size=BEST["batch_size"] * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

positives = float(y_train.sum())
negatives = float(len(y_train) - positives)
pos_weight = torch.tensor([negatives / max(positives, 1.0)], dtype=torch.float32, device=DEVICE)

final_model = LSTMClassifier(len(FEATURES), BEST["hidden_size"], BEST["num_layers"], BEST["dropout"]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(final_model.parameters(), lr=BEST["lr"], weight_decay=BEST["weight_decay"])

best_f2 = -1.0
best_state = None
training_history = []
patience_counter = 0

for epoch in range(1, MAX_EPOCHS + 1):
    final_model.train()
    batch_losses = []
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = final_model(x_batch.to(DEVICE))
        loss = criterion(logits, y_batch.to(DEVICE))
        loss.backward()
        optimizer.step()
        batch_losses.append(float(loss.item()))

    torch.save(final_model.state_dict(), CHECKPOINT_DIR / "lstm_local_latest.pt")

    final_model.eval()
    val_probabilities = []
    with torch.no_grad():
        for x_batch, _ in val_loader:
            val_probabilities.append(torch.sigmoid(final_model(x_batch.to(DEVICE))).cpu().numpy())
    val_probabilities = np.concatenate(val_probabilities)

    val_f2 = fbeta_score(y_val, (val_probabilities >= THRESHOLD).astype(int), beta=2, zero_division=0)
    val_f1 = f1_score(y_val, (val_probabilities >= THRESHOLD).astype(int), zero_division=0)

    training_history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(batch_losses)) if batch_losses else np.nan,
            "validation_f2": float(val_f2),
            "validation_f1": float(val_f1),
        }
    )

    if val_f2 > best_f2:
        best_f2 = float(val_f2)
        best_state = {key: value.detach().cpu().clone() for key, value in final_model.state_dict().items()}
        torch.save(best_state, CHECKPOINT_DIR / "lstm_local_best.pt")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            break

final_model.load_state_dict(best_state)

final_model.eval()
test_probabilities = []
with torch.no_grad():
    for x_batch, _ in test_loader:
        test_probabilities.append(torch.sigmoid(final_model(x_batch.to(DEVICE))).cpu().numpy())
test_probabilities = np.concatenate(test_probabilities)
test_predictions = (test_probabilities >= THRESHOLD).astype(int)

test_metrics = {
    "f2": float(fbeta_score(y_test, test_predictions, beta=2, zero_division=0)),
    "f1": float(f1_score(y_test, test_predictions, zero_division=0)),
    "precision": float(precision_score(y_test, test_predictions, zero_division=0)),
    "recall": float(recall_score(y_test, test_predictions, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, test_probabilities)),
}

training_history_df = pd.DataFrame(training_history)
test_predictions_df = full.iloc[test_used_idx][["datetime", "ACTUAL_POOL_PRICE", "pool_price_lead_2"]].copy()
test_predictions_df["y_true"] = y_test.astype(int)
test_predictions_df["y_prob"] = test_probabilities
test_predictions_df["y_pred"] = test_predictions

print(f"Best validation F2: {best_f2:.4f}")
print("Test metrics:")
for name, value in test_metrics.items():
    print(f"  {name}: {value:.4f}")

test_predictions_df.head()


In [ ]:

# 9. Save outputs
metrics_payload = {
    "target": TARGET,
    "spike_threshold": SPIKE_THRESHOLD,
    "threshold": THRESHOLD,
    "device": DEVICE,
    "feature_count": len(FEATURES),
    "best_params": BEST,
    "cv_mean_f2": round(float(cv_results["best_f2"].mean()), 4),
    "cv_std_f2": round(float(cv_results["best_f2"].std(ddof=0)), 4),
    "best_validation_f2": round(float(best_f2), 4),
    "test_metrics": {key: round(float(value), 4) for key, value in test_metrics.items()},
    "split_summary": split_summary.to_dict(orient="records"),
}

training_history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
cv_results.to_csv(OUTPUT_DIR / "cv_results.csv", index=False)
split_summary.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)
test_predictions_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
joblib.dump(scaler_final, OUTPUT_DIR / "scaler.joblib")

best_checkpoint = CHECKPOINT_DIR / "lstm_local_best.pt"
latest_checkpoint = CHECKPOINT_DIR / "lstm_local_latest.pt"
if best_checkpoint.exists():
    (OUTPUT_DIR / "best_model.pt").write_bytes(best_checkpoint.read_bytes())
if latest_checkpoint.exists():
    (OUTPUT_DIR / "latest_model.pt").write_bytes(latest_checkpoint.read_bytes())

with (OUTPUT_DIR / "metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(metrics_payload, handle, indent=2)

print(f"Saved notebook outputs to: {OUTPUT_DIR}")
print(f"Notebook path: {NOTEBOOK_DIR / 'lstm_aeso_local.ipynb'}")
